This question focuses on the multicollinearity problem

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

rng = np.random.default_rng(10)

(a) We create a linear model as shown in the code below. This corresponds to:
$$
y = \beta_0 + \beta_1x_1 + \beta_2x_2 + \varepsilon
$$
where coefficients $\beta_0 = 2$, $\beta_1 = 2$, $\beta_2 = 0.3$

In [ ]:
x1 = rng.uniform(0, 1, size=100)
x2 = 0.5 * x1 + rng.normal(size=100) / 10 # x2 is a function of x1
y = 2 + 2 * x1 + 0.3 * x2 + rng.normal(size=100) # we generate a response based on x1 and x2

(b) what is the correlation between x1 and x2? + Create a scatterplot showing the relationship between them.
- The correlation is 0.772324
- In the scatterplot, we can see a clear positive trend between x1 and x2, indicating a positive correlation. The points are clustered well together so the correlation is strong.

In [ ]:
# we create a pandas dataframe containing the values of x1 and corresponding values of x2, and compute the correlation between the two
df =pd.DataFrame({
    "x1": x1,
    "x2": x2,
})
df.corr() # 0.772324

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(x1, x2)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
plt.tight_layout()

(c) Fit a linear model to predict y with x1 and x2
- The F-statistic has a low p-value, so it shows that there is a relationship.
- $\hat{\beta}_0 = 1.95$ (this is close to the actual value, 2)
- $\hat{\beta}_1 = 1.61$ (this is far from the actual value, 2)
- $\hat{\beta}_2 = 0.94$ (this is far from the actual value, 0.3)
- The coefficient for intercept has a very low p-value and relatively low standard error
- The coefficients for x1 and x2 seem further from the true value and have relatively high standard errors.
- x1: The p-value of x1 is 0.003, so in the presence of x2, there is enough evidence to reject the null hypothesis $H_0:\beta_1=0$.
- x2: Even though it is part of the true relationship in generating response y, the p-value is 0.259, so there is not enough evidence to reject the null hypothesis $H_0: \beta_2=0$, in the presence of x1.

In [ ]:
X = pd.DataFrame({
    "intercept": np.ones(x1.shape),
    "x1": x1,
    "x2": x2,
})
result = sm.OLS(y, X).fit()
result.summary()

(d) Fit a linear model to predict y with only x1
- We can reject the null hypothesis $H_0: \beta_1 = 0$ and conclude that there is evidence of a relationship between y and x1.
- I will also note that the standard error for the estimate of $\beta_1$ has decreased significantly compared to the previous model.

In [ ]:
result2 = sm.OLS(y, X.drop(columns=['x2'])).fit()
result2.summary()

(e) Fit a linear model to predict y with only x2
- We can reject the null hypothesis $H_0: \beta_1 = 0$ for x2 because the p-value is very low
- In this setup the std error for the estimate of the coefficient for x2 is lower compared to the first model.

In [ ]:
result3 = sm.OLS(y, X.drop(columns=['x1'])).fit()
result3.summary()

(f) Explanation of results

In (c), we found that there is no evidence for x2 to be related to y. However, in (e), we found that there is in fact evidence for a relationship between x2 and y.

The response of y can be explained by both x1 and x2. We know this because we know the true relationship.

Because x1 and x2 can be predicted by each other (i.e. since they are strongly correlated), in the model with both of them the response y will be explained mostly by one of the variables, leaving the other one redundant. Hence, we see that x2 loses explanatory power in the 1st model but in the 3rd model when there is no x1 it is very much needed to explain response y.

(g) Investigation of new observation

In [ ]:
# code given by question
x1 = np.concat([x1, [0.1]])
x2 = np.concat([x2, [0.8]])
y = np.concat([y, [6]])

X_new = pd.DataFrame({
    "intercept": np.ones(x1.shape),
    "x1": x1,
    "x2": x2,
})
result_x1_x2 = sm.OLS(y, X_new).fit()
result_x1 = sm.OLS(y, X_new.drop(columns=["x2"])).fit()
result_x2 = sm.OLS(y, X_new.drop(columns=["x1"])).fit()

Response: y, Predictors: x1, x2
- coefficients for x1 and x2 changed dramatically, likely due to the multicollinearity
- standard errors for x1 and x2 went down
- p-values went down for x2 and went up for x1, this instability is also likely due to the multicollinearity
- note that the new coefficients are still not close to the true relationship, so this new data point didn't help the model

In [ ]:
pd.DataFrame({
    "orig coeff": result.params,
    "new coeff": result_x1_x2.params,
    "orig stderr": result.bse,
    "new stderr": result_x1_x2.bse,
    "orig pvalues": result.pvalues,
    "new pvalues": result_x1_x2.pvalues,
}, index=result_x1_x2.params.index)

Response: y, Predictors: x1
- coefficient for x1 went down
- std err for x1 went up (it also went up for the intercept)
- naturally, p-values went up as well

In [ ]:
pd.DataFrame({
    "orig coeff": result2.params,
    "new coeff": result_x1.params,
    "orig stderr": result2.bse,
    "new stderr": result_x1.bse,
    "orig pvalues": result2.pvalues,
    "new pvalues": result_x1.pvalues,
}, index=result_x1.params.index)

Response: y, Predictors: x2
- coefficient for x2 went up
- std error for x2 went down (it also went down for the intercept)
- naturally, p-values went down as well

In [ ]:
pd.DataFrame({
    "orig coeff": result3.params,
    "new coeff": result_x2.params,
    "orig stderr": result3.bse,
    "new stderr": result_x2.bse,
    "orig pvalues": result3.pvalues,
    "new pvalues": result_x2.pvalues,
}, index=result_x2.params.index)

I will also make some plots to check whether the new point is an outlier, and has high leverage since it has been able to affect the results quite significantly despite there being 100 other data points.

An outlier is a point for which the response $y_i$ is far from expected.
A high leverage point is one that has an unusual $x_i$ compared to the other points.

- The x1-x2 plot shows that it is outside of the expected range of predictor values, making it a *high leverage* point.
- The y-x1 plot shows that it is outside of the expected range of response values, making it an *outlier*. It is within the range of x1 values, so it's not a high leverage point in this case.
- The y-x2 plot shows that it has an unusual x2 value, making it a *high leverage* point. It does follow the trend of y values, however so it's not an outlier in this case.

Upon investigating the studentized residual vs leverage plot, we can further verify that it is a outlier with high leverage, this is why it was able to affect the results so much. Furthermore, it has a Cook's Distance > 1, so numerically, this is considered to be a very influential datapoint and should be investigated further as to why its values are like that.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3), ncols=4)
ax[0].scatter(x1, x2)
ax[0].scatter(x1[-1], x2[-1], c="orange") # highlight the new point
ax[0].set_xlabel("x1")
ax[0].set_ylabel("x2")

ax[1].scatter(x1, y)
ax[1].scatter(x1[-1], y[-1], c="orange") # highlight the new point
ax[1].set_xlabel("x1")
ax[1].set_ylabel("y")

ax[2].scatter(x2, y)
ax[2].scatter(x2[-1], y[-1], c="orange") # highlight the new point
ax[2].set_xlabel("x2")
ax[2].set_ylabel("y")

plt.tight_layout()

# plot of studentized resid against leverage for outlier / influential point analysis
resid_studentized = result_x1_x2.get_influence().resid_studentized_internal
leverage = result_x1_x2.get_influence().hat_matrix_diag
ax[3].scatter(leverage, resid_studentized)
ax[3].scatter(leverage[-1], resid_studentized[-1], c="orange")
ax[3].set_xlabel("leverage")
ax[3].set_ylabel("studentized residual")
# plot cook's distance = 1
x_cook = np.linspace(0.001, 0.4, 100)
y_cook = np.sqrt((1 * 3 * (1 - x_cook)) / x_cook)
ax[3].plot(x_cook, y_cook, c="k", ls="--", label="Cook's Distance = 1")
ax[3].set_ylim(-3.5, 3.5)
ax[3].legend()